## <center><u><span style="color:blue">Job Validator</span></u></center>
### <u><span style="color:coral">Background</span></u>:
#### It's become clear to me that not only are job applicants facing an uphill battle in setting themselves up for success with every job application, but they are also faced with an uneasy feeling that the job they're adapting their resume for... may not be legitimate.
#### According to <span style="color:blue">LinkedIn</span> itself, up to (<i>get this</i>) <span style="color:red">[6 out of every 10</span> jobs posted are fake](https://www.linkedin.com/pulse/reality-fake-job-listings-linkedin-what-you-need-know-arani-ramnc/).
#### That's just flippin' ridiculous.
#### Listing things like <span style="color:deeppink">Lead Generation</span>, <span style="color:deeppink">Company Image Boost</span>, and - of course - <span style="color:deeppink">Data Harvesting</span> as reasons to why companies do this type of stuff, it just becomes another layer of nausea the job seeker has to deal with in what is already a humiliating, arduous process.
#### So I figured, "What good is a a resume optimizer without a tool that checks for the legitimacy of the job postings?"
#### And here we are.
***
### <span style="color:blue">Step 1: Imports</span>
#### <u>These are the tools we'll need for this program to run. Loads of them, so here is a breakdown of what they do:
##### <b><span style="color:green">requests</span></b> - 
##### <b><span style="color:green">bs4</span></b> / <b><span style="color:green">BeautifulSoup</span></b> -
##### <b><span style="color:green">re</span></b> -
##### <b><span style="color:green">time</span></b> -
##### <b><span style="color:green">urllib.parse</span></b> / <b><span style="color:green">urlparse</span></b> -
##### <b><span style="color:green">typing</span></b> / <b><span style="color:green">Optional</span></b> -
##### <b><span style="color:green">fuzzywuzzy</span></b> / <b><span style="color:green">fuzz, process</span></b> -
##### <b><span style="color:green">selenium</span></b> / <b><span style="color:green">webdriver</span></b> -
##### <b><span style="color:green">selenium.webdriver.chrome.options</span></b> / <b><span style="color:green">Options</span></b> -


In [1]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urlparse, urljoin
from typing import Optional, List, Dict, Any
# no longer 'fuzzywuzzy'
from rapidfuzz import fuzz, process
# for Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# for Chrome
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options as ChromeOptions
from webdriver_manager.chrome import ChromeDriverManager
# for Firefox
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from webdriver_manager.firefox import GeckoDriverManager
# for Edge
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from webdriver_manager.microsoft import EdgeChromiumDriverManager

***
### <span style="color:blue">Step 2: Functions - A Broad Overview</span>
##### <b><u><span style="color:blue ; font-family:menlo">get_webdriver</span></u></b>
> ##### A function that sets up the bulk of the code to be used by either <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b>, <b><span style="color:cornflowerblue">Saf</span><span style="color:red">a</span><span style="color:cornflowerblue">ri</span></b>, <b><span style="color:blue">Ed</span><span style="color:chartreuse">ge</span></b>, or <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b>. 
##### <b><u><span style="color:blue ; font-family:menlo">is_valid_url</span></u></b>
> ##### A structure check to see if a given piece of text looks like a legitimate URL / web address. It's set up to receive that familiar string of characters indicative of a url (eg: 'https://www.linkedin.com'). If that string of characters looks good, great! If not, <b><span style="color:green ; font-family:menlo">False</span>.
##### <b><u><span style="color:blue ; font-family:menlo">find_careers_page</span></u></b> 
> ##### Kind of our bread-and-butter function in this process. It's a pretty complicated function (for me it was, anyways), and - full disclosure - I got some help from <span style="color:blue">Gemini</span> on redirects and how to handle them.
***


### <span style="color:blue">Step 2a: the 'get_webdriver' function</span>
> ##### because not everyone uses Chrome, this function sets up the user's chosen browser ("web_driver") to work with Selenium Webdriver, which is (<i>very generally speaking</i>) a tool that acts like a remote control for your tv: <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b> remotes work with <b><span style="color:red">Chr</span><span style="color:gold">o</span><span style="color:green">me</span></b> TVs, <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b> remotes work with <b><span style="color:darkorange">Fire</span><span style="color:purple">fox</span></b> TVs, etc.

In [2]:
def get_webdriver(browser_name: str): 
    """
    This function sets up the 'browser_name' in which the code will be running. 
    We're aiming at the four main browsers people use: Chrome, Safari, Edge, and Firefox
    """
    # because no browser is set up yet:
    driver = None
    if browser_name.lower() == "chrome":
        options = ChromeOptions()
        # run chrome in the background without showing user - prevent a headache
        options.add_argument("--headless")
        # since we're running the app on Render and we need Chrome to start porperly
        options.add_argument("--no-sandbox")
        # reduce crash-rate
        options.add_argument("--disable-dev-shm-usage")
        # download the chrome webdriver with the ChromeDriverManager - see imports
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    elif browser_name.lower() == "firefox":
        options = FirefoxOptions()
        options.add_argument("--headless")
        # download the firefox webdriver with GeckoDriverManager - see imports
        driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()), options=options)
    elif browser_name.lower() == "edge":
        options = EdgeOptions()
        # edge runs in the background the old-fashioned way where you have to declare 'headless'
        options.add_argument("--headless")
        # download the edge webdriver with EdgeChromiumDriverManager - see imports
        driver = webdriver.Edge(service=EdgeService(EdgeChromiumDriverManager().install()), options=options)
    # Safari's special: if you're running it, you have no other choice but to open it as a new window
    elif browser_name.lower() == "safari":
        driver = webdriver.Safari()
        print("Using Safari means a visible Safari browser will launch on your Mac.")
    else:
        raise ValueError(f"Sorry! {browser_name} is not supported. Try 'Chrome,' 'Firefox', 'Edge', or 'Safari' instead.")
    return driver

#### <span style="color:blue">Step 2b: the 'is_valid_url' function</span>
> ##### answers the question: am I getting a real url with my request?

In [3]:
def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

#### <span style="color:blue">Step 2c: the 'find_careers_page' function - doozy #1</span>
> ##### <b><u>Details regarding this function</u>:</b>
> ##### 1.) it lowercases company names, removes spaces, and strips special characters from the company's name;
> ##### 2.) it looks for websites that may not have that '.com' at the end (like a '.net,' or '.org');
> ##### 3.) makes the app look like an actual web browser for the server;
> ##### 4.) checks to make sure the URL is valid and what to do if it is not;
> ##### 5.) handles redirects, both server and client-side (thanks, <span style="color:blue">Gemini</span>!);
> ##### 6.) employs <b><span style="color:cornflowerblue">Beautiful</span><span style="color:goldenrod">Soup</span></b> to scrape the HTML / XML for any signs that the server is running any redirect code on my machine (client-side) - HTML or JavaScript; and
> ##### 7.) it creates a URL path and checks it for careers

In [34]:
def find_careers_page(company_name: str) -> Optional[str]:
    print(f"🔍 Searching for careers page for: {company_name}")
    
    # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
    slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())
    print(f"📝 Slugged company name: {slugged_company}")

    # not everything's a '.com'
    possible_base_urls = [f"https://www.{slugged_company}.com", 
                          f"https://www.{slugged_company}.net", 
                          f"https://www.{slugged_company}.org"]
    
    # typical career posting paths & subdomains
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "/about/careers", "/job-search"]
    possible_subdomains = ["careers.", "jobs."]

    # make the server believe we're one of the more popular clients / browsers
    # -> also, helps avoid blocks / captchas
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
            )
        }
    
    # First, try to find the main company website
    base_url_found = None
    for base_url_attempt in possible_base_urls:
        print(f"🌐 Trying base URL: {base_url_attempt}")
        if not is_valid_url(base_url_attempt):
            print(f"❌ Invalid URL format: {base_url_attempt}")
            continue
        
        try:
            response = requests.get(base_url_attempt, headers=headers, timeout=7)
            response.raise_for_status()
            print(f"✅ Found working base URL: {response.url}")
            base_url_found = response.url
            
            # Check if we got redirected to a careers page
            career_keyword_in_redirect = any(
                keyword in response.url.lower()
                for keyword in ["careers", "jobs", "opportunities"]
            )
            
            if career_keyword_in_redirect:
                print(f"🎯 Base URL redirected to careers page: {response.url}")
                return response.url

            # Check for meta refresh redirects
            soup = BeautifulSoup(response.text, "html.parser")
            meta_refresh = soup.find("meta", attrs={"http-equiv": re.compile(r"refresh", re.I)})
            
            if meta_refresh and "content" in meta_refresh.attrs:
                content = meta_refresh["content"]
                match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
                if match:
                    redirect_url = match.group(2)
                    
                    if not urlparse(redirect_url).netloc:
                        redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                    
                    if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                        print(f"🎯 Found meta refresh to careers: {redirect_url}")
                        return redirect_url

            # Check for JavaScript redirects
            script_tags = soup.find_all("script")
            for script in script_tags:
                if script.string:
                    js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    if js_match:
                        redirect_url = js_match.group(1)
                        if not urlparse(redirect_url).netloc:
                            redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                        
                        if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                            print(f"🎯 Found JS redirect to careers: {redirect_url}")
                            return redirect_url
            
            break  # Found working base URL, exit loop
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Failed to reach {base_url_attempt}: {e}")
            time.sleep(1)
            continue
    
    # If we found a base URL, try different career paths
    if base_url_found:
        print(f"🔍 Trying career paths for: {base_url_found}")
        base_domain = base_url_found.rstrip('/')
        
        # Try different paths
        for path in possible_paths:
            test_url = base_domain + path
            print(f"🌐 Trying: {test_url}")
            try:
                response = requests.get(test_url, headers=headers, timeout=5)
                response.raise_for_status()
                if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
                    print(f"✅ Found careers page: {response.url}")
                    return response.url
            except requests.exceptions.RequestException as e:
                print(f"❌ Failed: {e}")
                time.sleep(1)
        
        # Try subdomains
        for subdomain in possible_subdomains:
            test_url = base_domain.replace("www.", subdomain)
            print(f"🌐 Trying subdomain: {test_url}")
            try:
                response = requests.get(test_url, headers=headers, timeout=5)
                response.raise_for_status()
                if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
                    print(f"✅ Found careers subdomain: {response.url}")
                    return response.url
            except requests.exceptions.RequestException as e:
                print(f"❌ Failed: {e}")
                time.sleep(1)
    
    print("❌ Could not find careers page")
    return None

In [5]:
# def find_careers_page(company_name: str) -> Optional[str]:
#     print(f"🔍 Searching for careers page for: {company_name}")
#     # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
#     slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())
#     print(f"📝 Slugged company name: {slugged_company}")
#     # not everything's a '.com'
#     possible_base_urls = [f"https://www.{slugged_company}.com", 
#                           f"https://www.{slugged_company}.net", 
#                           f"https://www.{slugged_company}.org"]
    
#     # typical career posting paths & subdomains
#     possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "/about/careers", "/job-search"]
#     possible_subdomains = ["careers.", "jobs."]

#     # make the server believe we're one of the more popular clients / browsers
#     # -> also, helps avoid blocks / captchas
#     headers = {
#         "User-Agent": (
#             "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
#             "AppleWebKit/537.36 (KHTML, like Gecko) "
#             "Chrome/91.0.4472.124 Safari/537.36"
#             )
#         }
    
#     # loop for legitimacy and domain building
    
#     for base_url_attempt in possible_base_urls:
#         # stop the loop if the 'possible_base_url' seems wonky and skip to the try block
#         if not is_valid_url(base_url_attempt):
#             continue
#         # our plan if the 'possible_base_url' doesn't seem right
#         try:
#             # Send request to the main company URL and give it 7 seconds to respond
#             response = requests.get(base_url_attempt, headers=headers, timeout=7)
            
#             # No response after 7 seconds, check to see if we got an error message (like a 404 or 500 error)
#             response.raise_for_status()

#             # did the url redirect / somehow change after our request
#             we_got_redirected = response.url.lower() != base_url_attempt.lower()
            
#             # does that redirect have the career keywords we want?
#             career_keyword_in_redirect = any (
#                 keyword in response.url.lower()
#                 for keyword in ["careers", "jobs", "opportunities"]
#             )

#             # if that redirect has any of our 'career' keywords, then it's likely the page we want and we can stop searching
#             if we_got_redirected and career_keyword_in_redirect:
#                 return response.url

#             # check the HTML to see if server is running JavaScript redirects in my browser 
#             # -> important b/c 'requests' can't do JavaScript
#             soup = BeautifulSoup(response.text, "html.parser")

#             # get soup to find any <meta> refresh HTML tags (look like '<meta http-equiv="refresh">')
#             # -> we're looking for are 'http-equiv' that matches a case-insensitive pattern of 'refresh'
#             meta_refresh = soup.find("meta", attrs={"http-equiv": re.compile(r"refresh", re.I)})
            
#             # if soup finds a <meta> refresh tag with a 'content' attribute:
#             if meta_refresh and "content" in meta_refresh.attrs:
#                 # take that content text and store it as 'content'
#                 content = meta_refresh["content"]
#                 # find anything in 'content' that looks like 'url=whatever' and store it 
#                 match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
#                 # if we get a match, store it as 'redirect_url'
#                 if match:
#                     redirect_url = match.group(2)
                    
#                     # break the redirect down into its components; and 
#                     # change the relative URL path into to an absolute URL path
#                     # -> '.netloc' == 'network location" == domain
#                     if not urlparse(redirect_url).netloc:
#                         redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                    
#                     # If the redirected URL looks valid; AND
#                     # the redirect contains "careers" or "jobs", give us the redirect url
#                     if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
#                         return redirect_url

#             # JavaScript tag check -> Soup search for all occurences of '<script>' tags in the HTML 
#             script_tags = soup.find_all("script")

#             # loop through the tags we found
#             for script in script_tags:
#                 # check for and read the string content between the '<script></script>' tags
#                 # using regex to look to match the'window.location.href' string that indicates redirects
#                 if script.string:
#                     js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    
#                     # if there is a match, go to it (the redirect URL)
#                     if js_match:
#                         redirect_url = js_match.group(1)
#                         # change relative URLs to absolute URLs - like the '.netloc' note above
#                         if not urlparse(redirect_url).netloc:
#                             redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                        
#                         # If the redirected URL is valid and contains "careers" or "jobs", return it
#                         if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
#                             return redirect_url

#        # what to do about that bad url from the beginning (see 'try')
#         except requests.exceptions.RequestException:
            
#             # If the initial request to the base URL fails, try the very next base domain or path w/ a 1-second delay
#             time.sleep(1)
#             continue
#     # create a url with the main url and 'extend' the possible paths:
#     # -> turn 'https://www.bobsbeds.com' into 'https://www.bobsbeds.com/careers' 
#     potential_urls = [base_url_attempt + path for path in possible_paths]
    
#     # kind of like the above when adding paths to the base URL,
#     # but this time with subdomains
#     # -> building the 'potential_urls' list with '.extend' instead of '.append' so we can add ALL the URLs (not just one)
#     potential_urls.extend([
#         # replace 'www' with the subdomain - end up with something like 'https://careers.google.com' instead of 'https://www.google.com'
#         base_url_attempt.replace("www.", subdomain)
#         for subdomain in possible_subdomains
#         # 
#         if is_valid_url(base_url_attempt.replace("www.", subdomain) + possible_paths[0])
#     ])
#     potential_urls.append(f"https://{slugged_company}.com/careers")

#     # for loop to check possible career pages
#     for url in potential_urls:
#         # "if" the address is properly formatted
#         if is_valid_url(url):
#             try:
#                 response = requests.get(url, headers=headers, timeout=5)
#                 response.raise_for_status()
#                 if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
#                     return response.url
#             # if it is not...
#             except requests.exceptions.RequestException:
#                 time.sleep(1)
                
#     # if we can't find or architect an actual career page:
#     return None

### <span style="color:blue">Step 2d: the 'sanitize_title' function</span>
> ##### takes the job title character string and uses Regex to remove all the non-alphanumeric characters and whitespaces

In [6]:
def sanitize_title(title: str) -> str:
    # remove anything not a-z, A-Z, the digits 0-9, whitespaces, or pipes
    title = re.sub(r'[^a-zA-Z0-9\s\-]', '', title)
    # remove extra whitespace (eg: take "Bob's  Beds" and make it "bobsbeds" 
    title = re.sub(r'\s+', ' ', title).strip()
    # lowercase for URL
    return title.lower()

### <span style="color:blue">Step 2e: the 'validate_job_posting' function - doozy #2</span>
##### <b>This function takes the specific job title and company name, checks for that job posting in your preferred browser (defaults to Chrome), and gives us a response by:</b>
> ##### 1.) Using that extensive <span style="color:blue ; font-family:menlo">'find_careers_page'</span> function to find the most likely career page URL and letting us know if it was not found;
> ##### 2.) Incorporating the <span style="color:blue ; font-family:menlo">'get_webdriver'</span> function to allow you to use your preferred browser;
> ##### 3.) Slugs the career title we're looking ('Assistant Manager' becomes 'assistantmanager') for and scapes the URL we found for it; and
> ##### 4.) Creates a list of potential job titles found inside the common HTML tags and compares them to the job we're looking for using fuzzy matching and scoring job title similarities
> ##### <b>...</b>
***
## <span style="color:red">Whoops...</span>
***
### <b><span style="color:green"><i>Corrected Step 2e:</i></span> job validation functions (<span style="color:blue"><i>please see notes on the original function at the end of these working functions</i>)</span></b>
#### <b><u>This means a couple of changes in the original thought process as well</u>:</b>
> ##### 1.) The new approach is <b><span style="color:coral">API-focused</span></b>, no longer scraping each company's individual careers page for validation;
> ##### 2.) Anyone who's ever applied to a job knows that not all companies accept job applications through their own jobsites - they instead use third_party platforms / <b><span style="color:red">Applicant Tracking Systems (ATS Systems)</span></b> like '<span style="color:darkseagreen">Greenhouse</span></b>' or '<b><span style="color:darkseagreen">Lever</span></b>;'
> ##### 3.) Because of this, we'll have to identify / classify those URL patterns of the more popular ATS systems and decide what to do if those ATS patterns are not found - this would imply the company is indeed self-hosting their carers page, and we'll have to work with thier unique API; 
> ##### 4.) These things we're looking for - these patterns - are going to have to come from searching of "<b><span style="color:darkorange">iframe tags</span></b>" inside the <b><span style="color:blue">HTML</span></b>; tags that embed another mini-browser or mini-web page inside the page you're on - "a site within a site;"
> ##### 5.) When we find these <b><span style="color:darkorange">iframe tags</span></b>, we can jump in and out of them to extract whatever data they carry

#### In short, we can still try validating job postings by checking them against company websites, but we'll just have to go about it differently than originally planned.

### <span style="color:blue">Step 2e.A: Identify the Underlying</span> <span style="color:red">ATS</span> / <span style="color:blue">Hiring Platform</span>
> ##### A search for specific keywords in URLs and HTMLs, in this case from six (6) of the more popular <b><span style="color:red">ATS Systems</span></b>:
> ##### 1.) <b><span style="color:darkseagreen">Greenhouse</span></b>;
> ##### 2.) <b><span style="color:darkseagreen">Lever</span></b>;
> ##### 3.) <b><span style="color:chocolate">My</span><span style="color:darkblue">Work</span><span style="color:chocolate">day</span></b>;
> ##### 4.) <b><span style="color:darkslateblue">Ashbyhq</span></b>;
> ##### 5.) <b><span style="color:red">i</span>cims</b>; and
> ##### 6.) <b><span style="color:green">SmartRecuriters</span></b>
> ##### If it's not one of the above systems, we're going to assume the company is hosting / managing its own job board.

In [7]:
def guess_from_url(url: str) -> str:
    """
    This function lowercases URLS and parses through them for keywords pointing to popular ATS platforms.
    If no keywords are found, the URL is determined to be self-hosted.
    """
    url = url.lower()
    print(f"🔍 Analyzing URL for platform: {url}")
    
    if "greenhouse.io" in url:
        print("🌱 Detected: Greenhouse")
        return "greenhouse"
    elif "lever.co" in url:
        print("🎚️ Detected: Lever")
        return "lever"
    elif "myworkdayjobs.com" in url or "/wday/" in url or "workday.com" in url:
        print("💼 Detected: Workday")
        return "workday"
    elif "ashbyhq.com" in url:
        print("🎯 Detected: Ashby")
        return "ashby"
    elif "icims.com" in url:
        print("📊 Detected: iCIMS")
        return "icims"
    elif "smartrecruiters.com" in url:
        print("🧠 Detected: SmartRecruiters")
        return "smartrecruiters"
    else:
        print("🏠 Detected: Self-hosted/Unknown")
        return "self_hosted"

def guess_from_html(url: str) -> str:
    """
    Sends an HTTP GET request to the URL, gives it 10 seconds to respond,
    and searches for platform-specific keywords in the HTML.
    """
    print(f"🔍 Analyzing HTML content for platform indicators...")
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        html = res.text.lower()
        
        if "greenhouse" in html:
            print("🌱 Detected from HTML: Greenhouse")
            return "greenhouse"
        elif "lever" in html:
            print("🎚️ Detected from HTML: Lever")
            return "lever"
        elif "workday" in html or "myworkdayjobs" in html:
            print("💼 Detected from HTML: Workday")
            return "workday"
        elif "ashbyhq" in html:
            print("🎯 Detected from HTML: Ashby")
            return "ashby"
        elif "icims" in html:
            print("📊 Detected from HTML: iCIMS")
            return "icims"
        elif "smartrecruiters" in html:
            print("🧠 Detected from HTML: SmartRecruiters")
            return "smartrecruiters"
        else:
            print("🏠 No platform detected from HTML")
            return "self_hosted"
    except Exception as e:
        print(f"❌ Error analyzing HTML: {e}")
        return "self_hosted"


In [8]:
# # from the URL
# def guess_from_url(url: str) -> str:
#     """
#     This function lowercases URLS and parses through them for keywords pointing to popular ATS platforms.
#     If no keywords are found, the URL is determined to be self-hosted.
#     """
#     url = url.lower()
#     if "greenhouse.io" in url:
#         return "greenhouse"
#     elif "lever.co" in url:
#         return "lever"
#     elif "myworkdayjobs.com" in url or "/wday/" in url or "workday.com" in url:
#         return "workday"
#     elif "ashbyhq.com" in url:
#         return "ashby"
#     elif "icims.com" in url:
#         return "icims"
#     elif "smartrecruiters.com" in url:
#         return "smartrecruiters"
#     else:
#         return "self_hosted"

# # from the HTML
# def guess_from_html(url: str) -> str:
#     """
#     Sends an HTTP GET request to the URL, gives it 10 seconds to respond (the 'res'), lowercases the html, 
#     and searches for the indicated keywords. An
#     """
#     try:
#         res = requests.get(url, timeout=10)
#         html = res.text.lower()
#         if "greenhouse" in html:
#             return "greenhouse"
#         elif "lever" in html:
#             return "lever"
#         elif "workday" in html or "myworkdayjobs" in html:
#             return "workday"
#         elif "ashbyhq" in html:
#             return "ashby"
#         elif "icims" in html:
#             return "icims"
#         elif "smartrecruiters" in html:
#             return "smartrecruiters"
#         else:
#             return "self_hosted"
#     except Exception as e:
#         return f"error: {e}"

### <span style="color:blue">Step 2e.B: API Adapters / Bridges</span>
- ##### Individual adapters need to be built for consistent comms between the application and each site's API.
- ##### These adapters check for jobs on the job boards of each of the API endpoints of the listed <b><span style="color:red">ATS</span></b> platforms.
- ##### <b><span style="color:darkseagreen">Greenhouse</span></b> and <b><span style="color:darkseagreen">Lever</span></b> have public JSON job feeds and, thus, <b>predictable endpoint structures:</b>
    > ##### <b><span style="color:darkseagreen">Greenhouse</span></b> = https://boards.greenhouse.io/embed/job_board?for=company&format=json
    > ##### <b><span style="color:darkseagreen">Lever</span></b> = https://jobs.lever.co/company?format=json
-  ##### <b><span style="color:chocolate">My</span><span style="color:darkblue">Work</span><span style="color:chocolate">day</span></b> is a bit different - it has a <b>dynamic endpoint:</b>
    > you have to <b>build out</b> the API endpoint, and from there you can pull out job titles from the job tree structure ('<span style="color:black ; font-family:menlo"><b>.json()["jobPostings"]</span></b>' in the example). 
- ##### <b><span style="color:darkslateblue">Ashbyhq</span></b> embeds its job data inside an HTML tag.
   > ##### '<b>script</b>' is that specific HTML tag, so the connector we build for Ashby has to find that embedded data and extract it.
- ##### <b><span style="color:red">i</span>cims</b>
- ##### <b><span style="color:green">SmartRecuriters</span></b>. 

In [9]:
# For Greenhouse
def validate_greenhouse(slugged_company: str, job_title: str) -> str:
    """
    A function that searches the 'slugged' company's greenhouse public JSON API job board for a specific job title.
    """
    print(f"🌱 Validating with Greenhouse API for: {slugged_company}")
    url = f"https://boards.greenhouse.io/embed/job_board?for={slugged_company}&format=json"
    print(f"🌐 API URL: {url}")
    
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        
        jobs = res.json()
        if not jobs:
            return f"❌ No jobs found in Greenhouse for '{slugged_company}'"
        
        titles = [job['title'] for job in jobs if 'title' in job]
        print(f"📝 Found {len(titles)} job titles in Greenhouse")
        
        if not titles:
            return f"❌ No job titles found in Greenhouse data"
        
        # Sanitize the search title for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result
            # Find the original title that corresponds to the sanitized match
            original_match = titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match on Greenhouse: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match on Greenhouse: '{original_match}' ({score}%)"
            else:
                return f"❌ No strong match found on Greenhouse (best: '{original_match}' at {score}%)"
        else:
            return f"❌ No jobs matched that title on Greenhouse"

    except requests.exceptions.RequestException as e:
        return f"❌ Error connecting to Greenhouse API for '{slugged_company}': {e}"
    except Exception as e:
        return f"❌ Error processing Greenhouse data for '{slugged_company}': {e}"

In [10]:
# # For Greenhouse
# def validate_greenhouse(slugged_company: str, job_title: str) -> Optional[str]:
#     """
#     A function that searches the 'slugged' company's greenhouse public JSON API job board ('url') for a specific job title. 
#     """
#     # tap into the greenhouse widget
#     url = f"https://boards.greenhouse.io/embed/job_board?for={slugged_company}&format=json"
    
#     # try block for HTTP GET request and parsing
#     try:
#         res = requests.get(url, timeout=5)
        
#         # was the request successsful?
#         res.raise_for_status()
        
#         # if so, then make a list
#         jobs = res.json()
#         titles = [job['title'] for job in jobs if 'title' in job]
        
#         # does the 'sanitized' job we're looking for match anything on the 'potential_titles' list? 
#         # uses the fuzzywuzzy imports from above to identify similarities and gives us back a tuple with 
#         # 2 pieces of information: 'best_match' and 'best_score'

#         result = process.extractOne(job_title, titles, scorer=fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match on Greenhouse: '{match}' ({score}%)"
#             else:
#                 return f"⚠️ No strong match found on Greenhouse(best: '{match}' at {score}%)"
#         else:
#             return f"Sorry, no jobs matched that title."

#     except Exception as e:
#         return f"❌ Error querying Greenhouse for '{slugged_company}': {e}"

In [11]:
# For Lever - FIXED VARIABLE NAME
def validate_lever(slugged_company: str, job_title: str) -> str:
    """
    Search for the slugged company's listing on Lever.co for specific job titles.
    """
    print(f"🎚️ Validating with Lever API for: {slugged_company}")
    url = f"https://jobs.lever.co/{slugged_company}?format=json"
    print(f"🌐 API URL: {url}")
    
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        
        jobs = res.json()
        if not jobs:
            return f"❌ No jobs found in Lever for '{slugged_company}'"
        
        titles = [job['text'] for job in jobs if 'text' in job]
        print(f"📝 Found {len(titles)} job titles in Lever")
        
        if not titles:
            return f"❌ No job titles found in Lever data"
        
        # Sanitize the search title for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result
            # Find the original title that corresponds to the sanitized match
            original_match = titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match on Lever: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match on Lever: '{original_match}' ({score}%)"
            else:
                return f"❌ No strong match found on Lever (best: '{original_match}' at {score}%)"
        else:
            return f"❌ No jobs matched that title on Lever"

    except requests.exceptions.RequestException as e:
        return f"❌ Error connecting to Lever API for '{slugged_company}': {e}"
    except Exception as e:
        return f"❌ Error processing Lever data for '{slugged_company}': {e}"

In [12]:
# # For Lever
# def validate_lever(slugged_company: str, job_title: str) -> Optional[str]:
#     """
#     Search for the slugged company's listing on Lever.co for specific job titles.
#     """
#     url = f"https://jobs.lever.co/{slugged_company}?format=json"
#     # GET request / parse
#     try:
#         res = requests.get(url, timeout=5)

#         # success?
#         res.raise_for_status()
        
#         # make a list
#         jobs = res.json()
#         titles = [job['text'] for job in jobs if 'text' in job]
        
#         # scoring tuple
#         result = process.extractOne(job_title, titles, scorer=fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match on Lever: '{match}' ({score}%)"
#             else:
#                 return f"⚠️ No strong match found on Lever (best: '{match}' at {score}%)"

#     except Exception as e:
#         return f"❌ Error querying Lever for '{slugged_company}': {e}"

In [13]:
# For Workday
def validate_workday(careers_url: str, job_title: str) -> str:
    """
    Function to take the base URL of the company's Workday presence and
    dynamically build the Workday API endpoint.
    """
    print(f"💼 Validating with Workday API")
    print(f"🌐 Careers URL: {careers_url}")
    
    try:
        # Extract company identifier and path from URL
        # Example: https://apple.wd5.myworkdayjobs.com/en-US/Apple_Careers
        match = re.search(r'https://([^.]+)\.wd\d+\.myworkdayjobs\.com/[^/]+/([^/?]+)', careers_url)
        if not match:
            return f"❌ Could not parse Workday URL format: {careers_url}"
        
        company_id = match.group(1)
        path_slug = match.group(2)
        
        # Build API endpoint
        api_url = f"https://{company_id}.wd5.myworkdayjobs.com/wday/cxs/{company_id}/{path_slug}/jobs"
        print(f"🌐 API URL: {api_url}")

        res = requests.get(api_url, timeout=15)
        res.raise_for_status()
        
        data = res.json()
        jobs = data.get("jobPostings", [])
        
        if not jobs:
            return f"❌ No jobs found in Workday feed"
        
        titles = [job["title"] for job in jobs if "title" in job]
        print(f"📝 Found {len(titles)} job titles in Workday")
        
        if not titles:
            return f"❌ No job titles found in Workday data"
        
        # Sanitize titles for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)

        if result:
            match, score = result
            # Find the original title
            original_match = titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match in Workday: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in Workday: '{original_match}' ({score}%)"
            else:
                return f"❌ No strong match found in Workday (best: '{original_match}' at {score}%)"
        else:
            return f"❌ No match found in Workday"
            
    except requests.exceptions.RequestException as e:
        return f"❌ Error connecting to Workday API: {e}"
    except Exception as e:
        return f"❌ Workday processing error: {e}"

In [14]:
# # For Workday
# def validate_workday(careers_url: str, job_title: str) -> str:
#     """
#     Function to take the base URL of the company's Workday presense ('careers_url')
#     and the job title you're looking for to dynamically build out the Workday endpoint.

#     For example:
    
#     Apple uses Workday, and their career page URL looks like 'https://apple.wd5.myworkdayjobs.com/en-US/careercenter.'
#     That's all well and good, but not for our purposes. We need to get to the actual API endpoint data to get a JSON response
#     back for our job validator.

#     This function would make a 'request.get()' call to that URL, and Workday will send us back the actual endpoint:
#     https://apple.wd5.myworkdayjobs.com/wday/cxs/apple/careercenter/jobs
#     """
#     # 'try block' logic because URL parsing is pretty volatile
#     try:
#         # break it up into substrings (at the /) for the API
#         parts = careers_url.split("/")
#         # Also split at the '.'
#         subdomain = parts[2].split(".")[0]
#         # Assign that last item in the 'part' list - '/careercenter' - to a variable
#         path_slug = parts[-1]

#         # Step 2: Build API endpoint with the subdomain and path_slug
#         api_url = f"https://{subdomain}.wd5.myworkdayjobs.com/wday/cxs/{subdomain}/{path_slug}/jobs"

#         res = requests.get(api_url, timeout=10)
#         res.raise_for_status()
#         jobs = res.json().get("jobPostings", [])

#         titles = [job["title"] for job in jobs if "title" in job]
#         if not titles:
#             return "❌ No jobs found in Workday feed."
#         result = process.extractOne(job_title, titles, scorer=fuzz.token_sort_ratio)

#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match in Workday: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match in Workday: '{match}' ({score}%)"
#             else:
#                 return "❌ No good match found in Workday."
#         else:
#             return "❌ No good match found in Workday."
#     except Exception as e:
#         return f"❌ Workday adapter error: {e}"


In [15]:
# For Ashby
def validate_ashby(careers_url: str, job_title: str) -> str:
    print(f"🎯 Validating with Ashby")
    print(f"🌐 Careers URL: {careers_url}")
    
    try:
        res = requests.get(careers_url, timeout=15)
        res.raise_for_status()
        
        soup = BeautifulSoup(res.text, "html.parser")
        script = soup.find("script", string=re.compile("__ASHBY_CACHE__"))
        
        if not script:
            return f"❌ Could not locate Ashby job data in page"
        
        json_match = re.search(r'__ASHBY_CACHE__\s*=\s*(\{.*?\});', script.string, re.DOTALL)
        if not json_match:
            return f"❌ Could not extract Ashby JSON data"
        
        import json
        jobs_data = json.loads(json_match.group(1))
        jobs = jobs_data.get("jobs", [])
        
        if not jobs:
            return f"❌ No jobs found in Ashby data"
        
        titles = [job["title"] for job in jobs if "title" in job]
        print(f"📝 Found {len(titles)} job titles in Ashby")
        
        if not titles:
            return f"❌ No job titles found in Ashby data"
        
        # Sanitize titles for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result
            original_match = titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match in Ashby: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in Ashby: '{original_match}' ({score}%)"
            else:
                return f"❌ No strong match found in Ashby (best: '{original_match}' at {score}%)"
        else:
            return f"❌ No match found in Ashby"
            
    except requests.exceptions.RequestException as e:
        return f"❌ Error connecting to Ashby: {e}"
    except Exception as e:
        return f"❌ Ashby processing error: {e}"

In [16]:
# # For Ashby
# def validate_ashby(careers_url: str, job_title: str) -> str:
#     try:
#         res = requests.get(careers_url, timeout=10)
#         soup = BeautifulSoup(res.text, "html.parser")
#         script = soup.find("script", text=re.compile("__ASHBY_CACHE__"))
#         if not script:
#             return "❌ Could not locate Ashby job data."
#         json_text = re.search(r'__ASHBY_CACHE__\s*=\s*(\{.*?\});', script.string, re.DOTALL)
#         if not json_text:
#             return "❌ Could not extract Ashby JSON."
#         jobs_data = json.loads(json_text.group(1))
#         jobs = jobs_data.get("jobs", [])
#         titles = [job["title"] for job in jobs if "title" in job]
#         result = process.extractOne(job_title, titles, scorer = fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match in Ashby: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match in Ashby: '{match}' ({score}%)"
#             else:
#                 return "❌ No good match found in Ashby."
#     except Exception as e:
#         return f"❌ Ashby adapter error: {e}"

In [17]:
# For SmartRecruiters
def validate_smartrecruiters(careers_url: str, job_title: str) -> str:
    print(f"🧠 Validating with SmartRecruiters")
    print(f"🌐 Careers URL: {careers_url}")
    
    try:
        # Extract company ID from URL
        match = re.search(r'smartrecruiters.com/([^/]+)', careers_url)
        if not match:
            return f"❌ Could not parse SmartRecruiters URL format"
        
        company_id = match.group(1)
        api_url = f"https://api.smartrecruiters.com/v1/companies/{company_id}/postings"
        print(f"🌐 API URL: {api_url}")
        
        res = requests.get(api_url, timeout=15)
        res.raise_for_status()
        
        data = res.json()
        jobs = data.get("content", [])
        
        if not jobs:
            return f"❌ No jobs found in SmartRecruiters"
        
        titles = [job["name"] for job in jobs if "name" in job]
        print(f"📝 Found {len(titles)} job titles in SmartRecruiters")
        
        if not titles:
            return f"❌ No job titles found in SmartRecruiters data"
        
        # Sanitize titles for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result
            original_match = titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match in SmartRecruiters: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in SmartRecruiters: '{original_match}' ({score}%)"
            else:
                return f"❌ No strong match found in SmartRecruiters (best: '{original_match}' at {score}%)"
        else:
            return f"❌ No match found in SmartRecruiters"
            
    except requests.exceptions.RequestException as e:
        return f"❌ Error connecting to SmartRecruiters API: {e}"
    except Exception as e:
        return f"❌ SmartRecruiters processing error: {e}"

In [18]:
# # --- SmartRecruiters Adapter ---
# def validate_smartrecruiters(careers_url: str, job_title: str) -> str:
#     try:
#         match = re.search(r'smartrecruiters.com/([^/]+)/([^/]+)', careers_url)
#         if not match:
#             return "❌ Sorry. Something happened and we couldn't parse the SmartRecruiters URL."
#         slugged_company = match.group(1)
#         api_url = f"https://api.smartrecruiters.com/v1/companies/{slugged_company}/postings"
#         res = requests.get(api_url, timeout=10)
#         res.raise_for_status()
#         jobs = res.json().get("content", [])
#         titles = [job["name"] for job in jobs if "name" in job]
#         result = process.extractOne(job_title, titles, scorer=fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match in SmartRecruiters: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match in SmartRecruiters: '{match}' ({score}%)"
#             else:
#                 return "❌ No good match found in SmartRecruiters."
#         else:
#             return "❌ No match found in SmartRecruiters."
#     except Exception as e:
#         return f"❌ SmartRecruiters adapter error: {e}"

In [19]:
# For iCIMS
def validate_icims(careers_url: str, job_title: str) -> str:
    print(f"📊 Validating with iCIMS")
    print(f"🌐 Careers URL: {careers_url}")
    
    try:
        # Try multiple iCIMS API patterns
        parsed_base = re.match(r"^(https://[^/]+/[^/]+)", careers_url)
        if not parsed_base:
            return f"❌ Could not parse iCIMS base URL"

        base_url = parsed_base.group(1)
        
        # Try different API endpoints
        api_endpoints = [
            f"{base_url}/jobs/search.json",
            f"{base_url}/jobs.json",
            f"{base_url}/api/jobs"
        ]
        
        for api_url in api_endpoints:
            print(f"🌐 Trying API URL: {api_url}")
            try:
                response = requests.get(api_url, timeout=10)
                response.raise_for_status()
                
                data = response.json()
                jobs = data.get("jobs", []) or data.get("data", []) or data
                
                if jobs and isinstance(jobs, list):
                    titles = []
                    for job in jobs:
                        if isinstance(job, dict):
                            title = job.get("title") or job.get("jobTitle") or job.get("name")
                            if title:
                                titles.append(title)
                    
                    if titles:
                        print(f"📝 Found {len(titles)} job titles in iCIMS")
                        
                        # Sanitize titles for better matching
                        sanitized_job_title = sanitize_title(job_title)
                        sanitized_titles = [sanitize_title(title) for title in titles]
                        
                        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
                        
                        if result:
                            match, score = result
                            original_match = titles[sanitized_titles.index(match)]
                            
                            if score >= 85:
                                return f"✅ Found match in iCIMS: '{original_match}' ({score}%)"
                            elif score >= 70:
                                return f"⚠️ Partial match in iCIMS: '{original_match}' ({score}%)"
                            else:
                                return f"❌ No strong match found in iCIMS (best: '{original_match}' at {score}%)"
                        else:
                            return f"❌ No match found in iCIMS"
                
            except requests.exceptions.RequestException:
                continue
            except Exception:
                continue
        
        return f"❌ Could not access iCIMS API endpoints"
        
    except Exception as e:
        return f"❌ iCIMS processing error: {e}"

In [20]:
# # --- iCIMS Adapter ---
# def validate_icims(careers_url: str, job_title: str) -> str:
#     try:
#         # Attempt to use the known /jobs/search.json iCIMS pattern
#         parsed_base = re.match(r"^(https://[\w.-]+/[^/]+)", careers_url)
#         if not parsed_base:
#             return "❌ Could not parse iCIMS base URL."

#         search_url = parsed_base.group(1) + "/jobs/search.json"
#         response = requests.get(search_url, timeout=10)
#         response.raise_for_status()
#         data = response.json()
#         jobs = data.get("jobs", [])

#         titles = [job["title"] for job in jobs if "title" in job]
#         result = process.extractOne(job_title, titles, scorer=fuzz.token_sort_ratio)

#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match in iCIMS: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match in iCIMS: '{match}' ({score}%)"
#             else:
#                 return "❌ No good match found in iCIMS."
#         else:
#             return "❌ No match found in iCIMS."

#     except Exception as e:
#         return f"❌ iCIMS adapter error: {e}"

In [21]:
# Simple HTML Scraper
def validate_simple_html(careers_url: str, job_title: str) -> str:
    print(f"🔍 Validating with simple HTML scraping")
    print(f"🌐 URL: {careers_url}")
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        
        res = requests.get(careers_url, headers=headers, timeout=15)
        res.raise_for_status()
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        # Look for job titles in various HTML elements
        potential_titles = []
        
        # Common selectors for job titles
        selectors = [
            "h1", "h2", "h3", "h4",
            ".job-title", ".position-title", ".role-title",
            "[data-job-title]", "[data-position]",
            "a[href*='job']", "a[href*='position']", "a[href*='career']"
        ]
        
        for selector in selectors:
            elements = soup.select(selector)
            for element in elements:
                title_text = element.get_text(strip=True)
                # Filter out very short or very long text
                if 5 <= len(title_text) <= 100:
                    potential_titles.append(title_text)
        
        # Remove duplicates while preserving order
        seen = set()
        unique_titles = []
        for title in potential_titles:
            if title.lower() not in seen:
                seen.add(title.lower())
                unique_titles.append(title)
        
        print(f"📝 Found {len(unique_titles)} potential job titles in HTML")
        
        if not unique_titles:
            return f"❌ No potential job titles found in HTML content"
        
        # Sanitize titles for better matching
        sanitized_job_title = sanitize_title(job_title)
        sanitized_titles = [sanitize_title(title) for title in unique_titles]
        
        result = process.extractOne(sanitized_job_title, sanitized_titles, scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result
            original_match = unique_titles[sanitized_titles.index(match)]
            
            if score >= 85:
                return f"✅ Found match in HTML: '{original_match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in HTML: '{original_match}' ({score}%)"
            else:
                return "❌ No good HTML match found."
        else:
            return "❌ No good HTML match found"

    except Exception as e:
        return f"❌ HTML adapter error: {e}"
            

In [22]:
# # --- Simple HTML Scraper ---
# def validate_simple_html(careers_url: str, job_title: str) -> str:
#     try:
#         res = requests.get(careers_url, timeout=10)
#         soup = BeautifulSoup(res.text, "html.parser")
#         texts = [t.get_text(strip=True) for t in soup.find_all(["h2", "h3", "a", "p"]) if len(t.get_text(strip=True).split()) <= 20]
#         result = process.extractOne(job_title, texts, scorer=fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found close match in static HTML: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match found: '{match}' ({score}%)"
#             else:
#                 return "❌ No match found in static content."
#         else:
#                 return "❌ No match found in static content."
#     except Exception as e:
#         return f"❌ Error during HTML scrape: {e}"

In [23]:
# --- Improved Selenium Fallback ---
def validate_with_selenium(careers_url: str, job_title: str) -> str:
    driver = None
    try:
        options = ChromeOptions()
        # Enhanced Chrome options for better stability
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--disable-extensions')
        options.add_argument('--disable-images')
        options.add_argument('--disable-javascript')  # Faster loading for text content
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
        # Add timeout and page load strategy
        options.add_argument('--page-load-strategy=eager')
        options.set_capability('pageLoadStrategy', 'eager')
        
        service = ChromeService(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=options)
        driver.set_page_load_timeout(20)
        driver.implicitly_wait(5)
        
        driver.get(careers_url)
        
        # Wait for content with multiple strategies
        wait = WebDriverWait(driver, 15)
        try:
            # Try multiple selectors to catch different page structures
            wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
            wait.until(EC.any_of(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='job'], .job-title, .position")),
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1, h2, h3, h4")),
                EC.presence_of_element_located((By.CSS_SELECTOR, "a, p")),
            ))
        except TimeoutException:
            pass  # Continue anyway, page might have loaded
        
        # Enhanced content extraction
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Remove script and style elements
        for script in soup(["script", "style", "nav", "header", "footer"]):
            script.decompose()
        
        # Extract text from relevant elements with better filtering
        selectors = [
            'h1', 'h2', 'h3', 'h4', 'h5', 'h6',  # Headers
            'a[href*="job"]', 'a[href*="position"]', 'a[href*="career"]',  # Job links
            '.job-title', '.position-title', '.role-title',  # Common job title classes
            'p', 'span', 'div'  # General text elements
        ]
        
        texts = []
        for selector in selectors:
            elements = soup.select(selector)
            for element in elements:
                text = element.get_text(strip=True)
                if text and 3 <= len(text) <= 100 and len(text.split()) <= 25:  # Better length filtering
                    texts.append(text)
        
        # Remove duplicates while preserving order
        texts = list(dict.fromkeys(texts))
        
        if not texts:
            return "❌ No text content found on page."
        
        # Enhanced fuzzy matching with multiple scoring methods
        matches = []
        scorers = [fuzz.token_sort_ratio, fuzz.token_set_ratio, fuzz.partial_ratio]
        
        for scorer in scorers:
            result = process.extractOne(job_title, texts, scorer=scorer)
            if result:
                matches.append(result)
        
        # Get the best match across all scoring methods
        if matches:
            best_match = max(matches, key=lambda x: x[1])
            match, score = best_match
            
            if score >= 85:
                return f"✅ Found match using Selenium: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match using Selenium: '{match}' ({score}%)"
            else:
                # Try a more lenient search with individual words
                job_words = job_title.lower().split()
                for text in texts:
                    text_lower = text.lower()
                    word_matches = sum(1 for word in job_words if word in text_lower)
                    if word_matches >= len(job_words) * 0.6:  # 60% of words match
                        return f"⚠️ Keyword match using Selenium: '{text}' (word match)"
                
                return f"❌ No good match found. Best: '{match}' ({score}%)"
        else:
            return "❌ No matches found with Selenium."
            
    except TimeoutException:
        return "❌ Selenium timeout: Page took too long to load."
    except WebDriverException as e:
        return f"❌ Selenium WebDriver error: {str(e)[:100]}..."
    except Exception as e:
        return f"❌ Selenium error: {str(e)[:100]}..."
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass

In [24]:
# # --- Selenium Fallback ---
# def validate_with_selenium(careers_url: str, job_title: str) -> str:
#     try:
#         options = ChromeOptions()
#         options.add_argument('--headless')
#         options.add_argument('--no-sandbox')
#         options.add_argument('--disable-dev-shm-usage')
#         driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
#         driver.get(careers_url)
#         WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.CSS_SELECTOR, "a, h2, h3"))
#         )
#         soup = BeautifulSoup(driver.page_source, 'html.parser')
#         texts = [t.get_text(strip=True) for t in soup.find_all(['h2', 'h3', 'a', 'p']) if len(t.get_text(strip=True).split()) <= 20]
#         result = process.extractOne(job_title, texts, scorer=fuzz.token_sort_ratio)
#         if result:
#             match, score = result
#             if score >= 85:
#                 return f"✅ Found match using Selenium: '{match}' ({score}%)"
#             elif score >= 70:
#                 return f"⚠️ Partial match using Selenium: '{match}' ({score}%)"
#             else:
#                 return "❌ No match found with Selenium."
#         else:
#             return "❌ No match found with Selenium."
#     except Exception as e:
#         return f"❌ Selenium error: {e}"
#     finally:
#         driver.quit()

In [25]:
def validate_job_posting_api(company_name: str, job_title: str, platform: str = "auto", careers_url: Optional[str] = None) -> str:
    """
    Enhanced dispatcher with better error handling and platform detection.
    """
    if not job_title or not job_title.strip():
        return "❌ Job title cannot be empty."
    
    if not company_name or not company_name.strip():
        return "❌ Company name cannot be empty."
    
    slugged_company = company_name.lower().replace(" ", "").replace("-", "").replace("_", "")
    
    # Enhanced platform detection
    if platform == "auto" and careers_url:
        platform = guess_from_url(careers_url)
        if platform == "self_hosted":
            try:
                platform = guess_from_html(careers_url)
            except Exception as e:
                print(f"Warning: Could not analyze HTML for platform detection: {e}")
    
    # Platform-specific validation with error handling
    try:
        if platform == "greenhouse":
            return validate_greenhouse(slugged_company, job_title)
        elif platform == "lever":
            return validate_lever(slugged_company, job_title)
        elif platform == "workday":
            if not careers_url:
                return "❌ Workday requires a careers URL."
            return validate_workday(careers_url, job_title)
        elif platform == "ashby":
            if not careers_url:
                return "❌ Ashby requires a careers URL."
            return validate_ashby(careers_url, job_title)
        elif platform == "icims":
            if not careers_url:
                return "❌ iCIMS requires a careers URL."
            return validate_icims(careers_url, job_title)
        elif platform == "smartrecruiters":
            if not careers_url:
                return "❌ SmartRecruiters requires a careers URL."
            return validate_smartrecruiters(careers_url, job_title)
        elif platform == "self_hosted" and careers_url:
            # Try simple HTML first, then Selenium as fallback
            try:
                html_result = validate_simple_html(careers_url, job_title)
                if "✅" in html_result or ("⚠️" in html_result and "Partial match" in html_result):
                    return html_result
            except Exception as e:
                print(f"HTML validation failed, trying Selenium: {e}")
            
            return validate_with_selenium(careers_url, job_title)
        else:
            if careers_url:
                return f"❌ Unsupported platform '{platform}' for: {careers_url}"
            else:
                return f"❌ Could not detect platform. Please provide a careers URL."
                
    except Exception as e:
        return f"❌ Error during {platform} validation: {str(e)[:100]}..."


In [26]:
# # --- Dispatcher Function ---
# def validate_job_posting_api(company_name: str, job_title: str, platform: str = "auto", careers_url: Optional[str] = None) -> str:
#     slugged_company = company_name.lower().replace(" ", "")

#     if platform == "auto" and careers_url:
#         platform = guess_from_url(careers_url)
#         if platform == "self_hosted":
#             platform = guess_from_html(careers_url)

#     if platform == "greenhouse":
#         return validate_greenhouse(slugged_company, job_title)
#     elif platform == "lever":
#         return validate_lever(slugged_company, job_title)
#     elif platform == "workday":
#         return validate_workday(careers_url, job_title)
#     elif platform == "ashby":
#         return validate_ashby(careers_url, job_title)
#     elif platform == "icims":
#         return validate_icims(careers_url, job_title)
#     elif platform == "smartrecruiters":
#         return validate_smartrecruiters(careers_url, job_title)
#     elif platform == "self_hosted" and careers_url:
#         html_result = validate_simple_html(careers_url, job_title)
#         if "✅" in html_result or "⚠️" in html_result:
#             return html_result
#         return validate_with_selenium(careers_url, job_title)
#     else:
#         return f"❌ Could not detect or support platform for: {careers_url}"

In [27]:
# # Example usage
# if __name__ == "__main__":
#     company = "esri"
#     job = "Account Manager - AEC"
#     careers_page = "https://www.esri.com/en-us/about/careers/job-search"
#     print(validate_job_posting_api(company, job, careers_url=careers_page))


### <span style="color:blue">Step 2f: the 'run_validation' function</span>
> ##### Simply takes the user's input and guides the resulting functions

In [28]:
def run_validation(job_title: str, company_name: str, careers_url: Optional[str] = None, max_retries: int = 2) -> str:
    """
    High-level wrapper with retry logic for validating a job title against a company's careers page.
    """
    for attempt in range(max_retries + 1):
        try:
            result = validate_job_posting_api(
                company_name=company_name,
                job_title=job_title,
                careers_url=careers_url
            )
            
            # If we got a successful result, return it
            if "✅" in result or "⚠️" in result:
                return result
            
            # If it's the last attempt or a definitive failure, return the result
            if attempt == max_retries or "requires a careers URL" in result or "Unsupported platform" in result:
                return result
                
            print(f"Attempt {attempt + 1} failed, retrying...")
            
        except Exception as e:
            if attempt == max_retries:
                return f"❌ All attempts failed. Last error: {str(e)[:100]}..."
            print(f"Attempt {attempt + 1} failed with exception, retrying...")
    
    return "❌ All validation attempts failed."

In [29]:
# def run_validation(job_title: str, company_name: str, careers_url: Optional[str] = None) -> str:
#     """
#     High-level wrapper for validating a job title against a company's careers page.
#     Automatically infers the job board platform and dispatches to the correct adapter.
#     """
#     return validate_job_posting_api(
#         company_name=company_name,
#         job_title=job_title,
#         careers_url=careers_url
#     )


In [30]:
def validate_input(prompt: str, required: bool = True) -> str:
    """Helper function to validate user input."""
    while True:
        value = input(prompt).strip()
        if value or not required:
            return value if value else None
        print("This field is required. Please enter a value.")

### <span style="color:blue">Step 3: Keep This Modular</span>
> ##### make sure this code only runs when needed

In [33]:
if __name__ == "__main__":
    print("🔍 Enhanced Job Validator")
    print("=" * 40)
    
    try:
        company = validate_input("Enter the company name (e.g., Esri): ")
        job = validate_input("Enter the job title (e.g., Account Manager - AEC): ")
        careers_page = validate_input("Enter the careers page URL (optional, press Enter to skip): ", required=False)
        
        print(f"\n🔍 Validating '{job}' at '{company}'...")
        if careers_page:
            print(f"📄 Using careers page: {careers_page}")
        print("⏳ This may take a moment...\n")
        
        result = run_validation(job_title=job, company_name=company, careers_url=careers_page)
        print("=" * 40)
        print("RESULT:")
        print(result)
        
    except KeyboardInterrupt:
        print("\n\n❌ Validation cancelled by user.")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")

🔍 Enhanced Job Validator


Enter the company name (e.g., Esri):  Huntress
Enter the job title (e.g., Account Manager - AEC):  Sales Engineer
Enter the careers page URL (optional, press Enter to skip):  https://job-boards.greenhouse.io/huntress/jobs/6599652003?gh_src=010bae4f3us



🔍 Validating 'Sales Engineer' at 'Huntress'...
📄 Using careers page: https://job-boards.greenhouse.io/huntress/jobs/6599652003?gh_src=010bae4f3us
⏳ This may take a moment...

🔍 Analyzing URL for platform: https://job-boards.greenhouse.io/huntress/jobs/6599652003?gh_src=010bae4f3us
🌱 Detected: Greenhouse
🌱 Validating with Greenhouse API for: huntress
🌐 API URL: https://boards.greenhouse.io/embed/job_board?for=huntress&format=json
Attempt 1 failed, retrying...
🔍 Analyzing URL for platform: https://job-boards.greenhouse.io/huntress/jobs/6599652003?gh_src=010bae4f3us
🌱 Detected: Greenhouse
🌱 Validating with Greenhouse API for: huntress
🌐 API URL: https://boards.greenhouse.io/embed/job_board?for=huntress&format=json
Attempt 2 failed, retrying...
🔍 Analyzing URL for platform: https://job-boards.greenhouse.io/huntress/jobs/6599652003?gh_src=010bae4f3us
🌱 Detected: Greenhouse
🌱 Validating with Greenhouse API for: huntress
🌐 API URL: https://boards.greenhouse.io/embed/job_board?for=huntress&fo

In [ ]:
# if __name__ == "__main__":
#     print("🔍 Job Validator")
#     company = input("Enter the company name (e.g., Esri): ").strip()
#     job = input("Enter the job title (e.g., Account Manager - AEC): ").strip()
#     careers_page = input("Enter the careers page URL (optional, press Enter to auto-detect): ").strip() or None

#     print("\nValidating...\n")
#     result = run_validation(job_title=job, company_name=company, careers_url=careers_page)
#     print(result)


***
### <center><span style="color:red">OLD</span> <span style="color:blue">'validate_job_posting' function</span></center>
#### <i><span style="color:red">Dumped because it's not accurate</span></i>.
#### For instance, <span style="color:blue">LinkedIn</span> posted an "Account Manager - AEC" opening for a company called 'Esri' that was manually validated by going to the Esri career page directly. Ran the program, and it came back as "<span style="color:red">invalid</span>."
#### <b><u><span style="color:blue">Here's what that original, </span><span style="color:red">inaccurate</span> <span style="color:blue"> function taught me</u>:</span></b>
> ##### 1.) <b><span style="color:darkviolet">Beautiful</span><span style="color:gold">Soup</span></b> may not be the right tool for this because it scrapes too quickly. A lot of the jobsites load the careers in JavaScript <i>after</i> <b><span style="color:darkviolet">Beautiful</span><span style="color:gold">Soup</span></b> scrapes them, so I was getting back a lot of false negatives.
> ##### 2.) Job listings are sometimes listed inside <b>'iframe'</b> tags, which are like mini-browsers inside the web page. <b><span style="color:darkviolet">Beautiful</span><span style="color:gold">Soup</span></b> cannot get the content from inside those frames - it'll give you the <b>iframe</b> tag itself, but that's it. We don't need tags. We need job descriptions.
> ##### 3.) You can add wait times to scraping - you can tell your code to wait for a page to load before executing. Tried that, but didn't improve the application's accuracy.
***

In [ ]:

# def validate_job_posting(job_title: str, company_name: str, browser_name: str = "chrome") -> str:
#     """
#     Checks if the given job_title exists / can be closely matched with any listings on the company's job website,
#     and returns a percentage confidence rating.
#     """
#     careers_url = find_careers_page(company_name)
#     if not careers_url:
#         # 'return' to stop the app since we couldn't find any career page
#         return "Sorry. We couldn't automatically find the company's careers page. You may have to check manually."

#     # try block to run the 'get_webdriver' function to initialize the driver that was returned
#     # in this function, it defaults to Chrome
#     try:
#         driver = get_webdriver(browser_name)
#     # in case we get that ValueError from 'get_webdriver:'
#     except ValueError as e:
#         return f"❌ Browser setup error: {e}"
#     # if there are any other problems - like the manager can't download the driver:
#     except Exception as e:
#         return f"❌ Could not initialize WebDriver: {e}. Make sure the correct WebDriver is installed and in your PATH."

#     # try block tells the driver - if successfully initialized - to go to the careers URL we found
#     # get BeautifulSoup to look it over for potential job titles
#     try:
#         driver.get(careers_url)

#         # JavaScript-rendered job listings load AFTER the page finishes loading
#         # introducing a wait time for everything to load before BeautifulSoup does its parsing
#         WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.CSS_SELECTOR, "a, h2, h3"))
#         )

#         # now scrape with BeautifulSoup
#         soup = BeautifulSoup(driver.page_source, "html.parser")

#         job_title_sanitized = sanitize_title(job_title)

#         # create an empty list of potential job_titles against which we'll be checking our desired job
#         potential_titles = []

#         # go through the common HTML tags
#         for tag in soup.find_all(["h2", "h3", "a", "p", "div"]):
#             text = tag.get_text(separator=' ', strip=True)
#             # if the number of the words in the tag is more than 20, it's probably not a job title tag
#             # the number '20' was suggested by Google
#             if len(text.split()) <= 20:
#                 # add that job title to our list
#                 potential_titles.append(sanitize_title(text))

#         # empty-results handling
#         if not potential_titles:
#             return f"Sorry. No job titles were found to match against. It may be due very latent rendering.\nCareers page: {careers_url}"

#         # does the 'sanitized' job we're looking for match anything on the 'potential_titles' list? 
#         # uses the fuzzywuzzy imports from above to identify similarities and gives us back a tuple with 
#         # 2 pieces of information: 'best_match' and 'best_score'
#         best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

#         # 'best_score' from the 'fuzz.token_sort_ratio'
#         if best_score >= 85:
#             return f"✅ Looks legit! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
#         else:
#             return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

#     except Exception as e:
#         return f"❌ Seems like an error happened while we were scraping: {e}"

#     # finally, close the browser
#     finally:
#         # if the driver was, indeed, successfully initialized:
#         if driver: 
#             driver.quit()

### <span style="color:red">Old</span> '<span style="color:blue ; font-family:menlo">run_validation</span>' <span style="color:blue">function:</span>
> ##### Used the old, inaccurate '<b><span style="color:blue ; font-family:menlo">validate_job_posting</span></b>' function above

```python
def run_validation():
    job_title_input = input("Enter the job title you want to check from the posting: ")
    company_name_input = input("Enter the name of the company: ")
    validation_result = validate_job_posting(job_title_input, company_name_input)
    print("\nValidation Result:")
    print(validation_result)
```

***
### <center><span style="color:red">&darr; &darr; &darr;</span> <span style="color:blue">Other attempts / tries / failures</span> <span style="color:red"> &darr; &darr; &darr;</span></center>

In [ ]:
# import requests
# from bs4 import BeautifulSoup
# import re
# import time
# from urllib.parse import urlparse
# from typing import Optional
# from fuzzywuzzy import fuzz, process
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options

# def is_valid_url(url: str) -> bool:
#     try:
#         result = urlparse(url)
#         return all([result.scheme, result.netloc])
#     except ValueError:
#         return False

# def find_careers_page(company_name: str) -> Optional[str]:
    
#     # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
#     slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())

#     # not everything's a '.com'
#     possible_base_urls = [f"https://www.{slugged_company}.com", 
#                           f"https://www.{slugged_company}.net", 
#                           f"https://www.{slugged_company}.org"]
    
#     # typical career posting paths & subdomains
#     possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
#     possible_subdomains = ["careers.", "jobs."]

#     # make the server believe we're one of the more popular clients / browsers
#     # -> also, helps avoid blocks / captchas
#         headers = {
#         'User-Agent': (
#             "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
#             "AppleWebKit/537.36 (KHTML, like Gecko) "
#             "Chrome/91.0.4472.124 Safari/537.36"
#             )
#         }
    
#     # loop for legitimacy and domain building
    
#     for base_url_attempt in possible_base_urls:
#         # stop the loop if the 'possible_base_url' seems wonky and skip to the try block
#         if not is_valid_url(base_url_attempt):
#             continue
#         # our plan if the 'possible_base_url' doesn't seem right
#         try:
#             # Send request to the main company URL and give it 7 seconds to respond
#             response = requests.get(base_url_attempt, headers=headers, timeout=7)
            
#             # No response after 7 seconds, check to see if we got an error message (like a 404 or 500 error)
#             response.raise_for_status()

#             # did the url redirect / somehow change after our request
#             we_got_redirected = response.url.lower() != base_url_attempt.lower()
            
#             # does that redirect have the career keywords we want?
#             career_keyword_in_redirect = any (
#                 keyword in response.url.lower()
#                 for keyord in ["careers", "jobs", "opportunities"]
#             )

#             # if that redirect has any of our 'career' keywords, then it's likely the page we want and we can stop searching
#             if we_got_redirected and career_keyword_in_redirect:
#                 return response.url

#             # check the HTML to see if server is running JavaScript redirects in my browser 
#             # -> important b/c 'requests' can't do JavaScript
#             soup = BeautifulSoup(response.text, "html.parser")

#             # get soup to find any <meta> refresh HTML tags (look like '<meta http-equiv="refresh">')
#             # -> we're looking for are 'http-equiv' that matches a case-insensitive pattern of 'refresh'
#             meta_refresh = soup.find("meta", attrs={"http-equiv": re.compile(r"refresh", re.I)})
            
#             # if soup finds a <meta> refresh tag with a 'content' attribute:
#             if meta_refresh and 'content' in meta_refresh.attrs:
#                 # take that content text and store it as 'content'
#                 content = meta_refresh["content"]
#                 # find anything in 'content' that looks like 'url=whatever' and store it 
#                 match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
#                 # if we get a match, store it as 'redirect_url'
#                 if match:
#                     redirect_url = match.group(2)
                    
#                     # break the redirect down into its components; and 
#                     # change the relative URL path into to an absolute URL path
#                     # -> '.netloc' == 'network location" == domain
#                     if not urlparse(redirect_url).netloc:
#                         redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                    
#                     # If the redirected URL looks valid; AND
#                     # the redirect contains "careers" or "jobs", give us the redirect url
#                     if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]): [cite: 3]
#                         return redirect_url

#             # JavaScript tag check -> Soup search for all occurences of '<script>' tags in the HTML 
#             script_tags = soup.find_all('script')

#             # loop through the tags we found
#             for script in script_tags:
#                 # check for and read the string content between the '<script></script>' tags
#                 # using regex to look to match the'window.location.href' string that indicates redirects
#                 if script.string:
#                     js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    
#                     # if there is a match, go to it (the redirect URL)
#                     if js_match:
#                         redirect_url = js_match.group(1)
#                         # change relative URLs to absolute URLs - like the '.netloc' note above
#                         if not urlparse(redirect_url).netloc:
#                             redirect_url = requests.compat.urljoin(base_url_attempt, redirect_url)
                        
#                         # If the redirected URL is valid and contains "careers" or "jobs", return it
#                         if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]): [cite: 3]
#                             return redirect_url

#        # what to do about that bad url from the beginning (see 'try')
#         except requests.exceptions.RequestException:
            
#             # If the initial request to the base URL fails, try the very next base domain or path w/ a 1-second delay
#             time.sleep(1)
#             continue
#     # create a url with the main url and 'extend' the possible paths:
#     # -> turn 'https://www.bobsbeds.com' into 'https://www.bobsbeds.com/careers' 
#     potential_urls = [base_url + path for path in possible_paths]
    
#     # kind of like the above when adding paths to the base URL,
#     # but this time with subdomains

# def sanitize_title(title: str) -> str:
#     title = re.sub(r'[^a-zA-Z0-9\s\-\|]', '', title)
#     title = re.sub(r'\s+', ' ', title).strip()
#     return title.lower()

# def validate_job_posting(job_title: str, company_name: str) -> str:
#     careers_url = find_careers_page(company_name)
#     if not careers_url:
#         return "Could not automatically find the company's careers page. Please check manually."

#     options = Options()
#     options.headless = True
#     driver = webdriver.Chrome(options=options)

#     try:
#         driver.get(careers_url)
#         soup = BeautifulSoup(driver.page_source, "html.parser")

#         job_title_sanitized = sanitize_title(job_title)
#         potential_titles = []

#         for tag in soup.find_all(["h2", "h3", "a", "p", "div"]):
#             text = tag.get_text(separator=' ', strip=True)
#             if len(text.split()) <= 20:
#                 potential_titles.append(sanitize_title(text))

#         best_match, best_score = process.extractOne(job_title_sanitized, potential_titles, scorer=fuzz.token_sort_ratio)

#         if best_score >= 85:
#             return f"✅ Fuzzy match found! '{job_title}' ≈ '{best_match}' ({best_score}% match)\nCareers page: {careers_url}"
#         else:
#             return f"⚠️ No close matches found for '{job_title}' (best: '{best_match}' at {best_score}% similarity).\nCheck manually: {careers_url}"

#     except Exception as e:
#         return f"❌ An error occurred during scraping: {e}"

#     finally:
#         driver.quit()

# def run_validation():
#     job_title_input = input("Enter the job posting title from LinkedIn: ")
#     company_name_input = input("Enter the name of the company: ")
#     validation_result = validate_job_posting(job_title_input, company_name_input)
#     print("\nValidation Result:")
#     print(validation_result)

# if __name__ == "__main__":
#     run_validation()


In [ ]:
# def find_careers_page(company_name: str) -> Optional[str]:
#     company_name_lower = company_name.lower().replace(' ', '')
#     base_url = f"https://www.{company_name_lower}.com"
#     possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "about/careers#open-positions"]
#     possible_subdomains = ["careers.", "jobs."]

#     potential_urls = [base_url + path for path in possible_paths]
#     potential_urls.extend([
#         base_url.replace("www.", subdomain)
#         for subdomain in possible_subdomains
#         if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
#     ])
#     potential_urls.append(f"https://{company_name_lower}.com/careers")

#     headers = {
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64) AppleWebKit/537.36 Chrome/91.0.4472.124 Safari/537.36"
#     }

#     for url in potential_urls:
#         if is_valid_url(url):
#             try:
#                 response = requests.get(url, headers=headers, timeout=5)
#                 response.raise_for_status()
#                 if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
#                     return response.url
#             except requests.exceptions.RequestException:
#                 time.sleep(1)

#     return None

### 

### 

In [ ]:
#     # -> building the 'potential_urls' list with '.extend' instead of '.append' so we can add ALL the URLs (not just one)
#     potential_urls.extend([
#         # replace 'www' with the subdomain - end up with something like 'https://careers.google.com' instead of 'https://www.google.com'
#         base_url.replace("www.", subdomain)
#         for subdomain in possible_subdomains
#         # 
#         if is_valid_url(base_url.replace("www.", subdomain) + possible_paths[0])
#     ])
#     potential_urls.append(f"https://{company_name_lower}.com/careers")

#     # for loop to check possible career pages
#     for url in potential_urls:
#         # "if" the address is properly formatted
#         if is_valid_url(url):
#             try:
#                 response = requests.get(url, headers=headers, timeout=5)
#                 response.raise_for_status()
#                 if any(k in response.url.lower() or k in response.text.lower() for k in ["careers", "jobs"]):
#                     return response.url
#             # if it is not...
#             except requests.exceptions.RequestException:
#                 time.sleep(1)

#     return None



In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
import json
from urllib.parse import urlparse, urljoin
from typing import Optional
# no longer 'fuzzywuzzy'
from rapidfuzz import fuzz, process
# for Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# for Chrome
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options as ChromeOptions
from webdriver_manager.chrome import ChromeDriverManager
# for Firefox
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from webdriver_manager.firefox import GeckoDriverManager
# for Edge
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.edge.options import Options as EdgeOptions
from webdriver_manager.microsoft import EdgeChromiumDriverManager

# Global debug flag
DEBUG = True

def debug_print(message: str):
    """Print debug messages if DEBUG is enabled"""
    if DEBUG:
        print(f"[DEBUG] {message}")

def get_webdriver(browser_name: str): 
    """
    This function sets up the 'browser_name' in which the code will be running. 
    We're aiming at the four main browsers people use: Chrome, Safari, Edge, and Firefox
    """
    # because no browser is set up yet:
    driver = None
    if browser_name.lower() == "chrome":
        options = ChromeOptions()
        # run chrome in the background without showing user - prevent a headache
        options.add_argument("--headless")
        # since we're running the app on Render and we need Chrome to start properly
        options.add_argument("--no-sandbox")
        # reduce crash-rate
        options.add_argument("--disable-dev-shm-usage")
        # download the chrome webdriver with the ChromeDriverManager - see imports
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    elif browser_name.lower() == "firefox":
        options = FirefoxOptions()
        options.add_argument("--headless")
        # download the firefox webdriver with GeckoDriverManager - see imports
        driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()), options=options)
    elif browser_name.lower() == "edge":
        options = EdgeOptions()
        # edge runs in the background the old-fashioned way where you have to declare 'headless'
        options.add_argument("--headless")
        # download the edge webdriver with EdgeChromiumDriverManager - see imports
        driver = webdriver.Edge(service=EdgeService(EdgeChromiumDriverManager().install()), options=options)
    # Safari's special: if you're running it, you have no other choice but to open it as a new window
    elif browser_name.lower() == "safari":
        driver = webdriver.Safari()
        print("Using Safari means a visible Safari browser will launch on your Mac.")
    else:
        raise ValueError(f"Sorry! {browser_name} is not supported. Try 'Chrome,' 'Firefox', 'Edge', or 'Safari' instead.")
    return driver

def is_valid_url(url: str) -> bool:
    try:
        result = urlparse(url)
        return all([result.scheme, result.netloc])
    except ValueError:
        return False

def find_careers_page(company_name: str) -> Optional[str]:
    debug_print(f"Looking for careers page for: {company_name}")
    
    # slug company names into the domain - turn 'Bob's Beds' into 'bobsbeds'
    slugged_company = re.sub(r"[^a-z0-9]", "", company_name.lower())
    debug_print(f"Slugged company name: {slugged_company}")

    # not everything's a '.com'
    possible_base_urls = [
        f"https://www.{slugged_company}.com", 
        f"https://www.{slugged_company}.net", 
        f"https://www.{slugged_company}.org",
        f"https://{slugged_company}.com"
    ]
    
    # typical career posting paths & subdomains
    possible_paths = ["/careers", "/jobs", "/about-us/careers", "/join-us", "/careers-jobs", "/about/careers", "/job-search", "/work-with-us"]
    possible_subdomains = ["careers.", "jobs."]

    # make the server believe we're one of the more popular clients / browsers
    # -> also, helps avoid blocks / captchas
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" 
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    
    # First, try to find a working base URL
    working_base_url = None
    
    for base_url_attempt in possible_base_urls:
        debug_print(f"Trying base URL: {base_url_attempt}")
        
        if not is_valid_url(base_url_attempt):
            debug_print(f"Invalid URL format: {base_url_attempt}")
            continue
            
        try:
            # Send request to the main company URL and give it 7 seconds to respond
            response = requests.get(base_url_attempt, headers=headers, timeout=7)
            
            # Check if we got a successful response
            response.raise_for_status()
            working_base_url = base_url_attempt
            debug_print(f"Found working base URL: {working_base_url}")

            # Check if we got redirected to a careers page
            if response.url.lower() != base_url_attempt.lower():
                debug_print(f"Got redirected to: {response.url}")
                career_keywords = ["careers", "jobs", "opportunities"]
                if any(keyword in response.url.lower() for keyword in career_keywords):
                    debug_print(f"Redirect contains career keywords, returning: {response.url}")
                    return response.url

            # Check for meta refresh redirects
            soup = BeautifulSoup(response.text, "html.parser")
            meta_refresh = soup.find("meta", attrs={"http-equiv": re.compile(r"refresh", re.I)})
            
            if meta_refresh and "content" in meta_refresh.attrs:
                content = meta_refresh["content"]
                match = re.search(r'url=(["\']?)(.*?)\1', content, re.I)
                if match:
                    redirect_url = match.group(2)
                    
                    # Convert relative URL to absolute URL
                    if not urlparse(redirect_url).netloc:
                        redirect_url = urljoin(base_url_attempt, redirect_url)
                    
                    if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                        debug_print(f"Found meta refresh redirect to careers: {redirect_url}")
                        return redirect_url

            # Check for JavaScript redirects
            script_tags = soup.find_all("script")
            for script in script_tags:
                if script.string:
                    js_match = re.search(r'window\.location\.href\s*=\s*["\'](.*?)["\']', script.string)
                    if js_match:
                        redirect_url = js_match.group(1)
                        if not urlparse(redirect_url).netloc:
                            redirect_url = urljoin(base_url_attempt, redirect_url)
                        
                        if is_valid_url(redirect_url) and any(k in redirect_url.lower() for k in ["careers", "jobs"]):
                            debug_print(f"Found JS redirect to careers: {redirect_url}")
                            return redirect_url
            
            # If we found a working base URL, break out to try specific paths
            break
            
        except requests.exceptions.RequestException as e:
            debug_print(f"Request failed for {base_url_attempt}: {e}")
            time.sleep(1)
            continue
    
    # If no working base URL found, return None
    if not working_base_url:
        debug_print("No working base URL found")
        return None
    
    # Now try specific career paths with the working base URL
    potential_urls = []
    
    # Add paths to the working base URL
    for path in possible_paths:
        potential_urls.append(working_base_url + path)
    
    # Add subdomain variations
    for subdomain in possible_subdomains:
        subdomain_url = working_base_url.replace("www.", subdomain)
        potential_urls.append(subdomain_url)
        # Also try subdomain + paths
        for path in possible_paths:
            potential_urls.append(subdomain_url + path)
    
    debug_print(f"Testing {len(potential_urls)} potential career URLs")
    
    # Test each potential URL
    for url in potential_urls:
        debug_print(f"Testing career URL: {url}")
        
        if is_valid_url(url):
            try:
                response = requests.get(url, headers=headers, timeout=5)
                response.raise_for_status()
                
                # Check if the response contains career-related content
                career_keywords = ["careers", "jobs", "opportunities", "positions", "openings"]
                url_has_keywords = any(k in response.url.lower() for k in career_keywords)
                content_has_keywords = any(k in response.text.lower() for k in career_keywords)
                
                if url_has_keywords or content_has_keywords:
                    debug_print(f"Found careers page: {response.url}")
                    return response.url
                    
            except requests.exceptions.RequestException as e:
                debug_print(f"Failed to access {url}: {e}")
                time.sleep(1)
                continue
                
    debug_print("No careers page found")
    return None

def sanitize_title(title: str) -> str:
    # remove anything not a-z, A-Z, the digits 0-9, whitespaces, or hyphens
    title = re.sub(r'[^a-zA-Z0-9\s\-]', '', title)
    # remove extra whitespace
    title = re.sub(r'\s+', ' ', title).strip()
    # lowercase for comparison
    return title.lower()

def guess_from_url(url: str) -> str:
    """
    This function lowercases URLs and parses through them for keywords pointing to popular ATS platforms.
    """
    url = url.lower()
    debug_print(f"Guessing platform from URL: {url}")
    
    if "greenhouse.io" in url:
        return "greenhouse"
    elif "lever.co" in url:
        return "lever"
    elif "myworkdayjobs.com" in url or "/wday/" in url or "workday.com" in url:
        return "workday"
    elif "ashbyhq.com" in url:
        return "ashby"
    elif "icims.com" in url:
        return "icims"
    elif "smartrecruiters.com" in url:
        return "smartrecruiters"
    else:
        return "self_hosted"

def guess_from_html(url: str) -> str:
    """
    Analyze HTML content to detect ATS platform
    """
    debug_print(f"Analyzing HTML content for platform detection: {url}")
    
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        html = res.text.lower()
        
        # Check for platform indicators in HTML
        if "greenhouse" in html or "greenhouse.io" in html:
            return "greenhouse"
        elif "lever" in html or "lever.co" in html:
            return "lever"
        elif "workday" in html or "myworkdayjobs" in html:
            return "workday"
        elif "ashbyhq" in html or "ashby" in html:
            return "ashby"
        elif "icims" in html:
            return "icims"
        elif "smartrecruiters" in html:
            return "smartrecruiters"
        else:
            return "self_hosted"
            
    except Exception as e:
        debug_print(f"Error analyzing HTML: {e}")
        return "self_hosted"

def validate_greenhouse(slugged_company: str, job_title: str) -> str:
    """
    Search Greenhouse API for job title
    """
    debug_print(f"Validating with Greenhouse for company: {slugged_company}")
    
    url = f"https://boards.greenhouse.io/embed/job_board?for={slugged_company}&format=json"
    debug_print(f"Greenhouse URL: {url}")
    
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        
        jobs = res.json()
        debug_print(f"Found {len(jobs)} jobs in Greenhouse")
        
        if not jobs:
            return "❌ No jobs found in Greenhouse feed"
        
        titles = [job.get('title', '') for job in jobs if job.get('title')]
        debug_print(f"Job titles found: {titles[:5]}...")  # Show first 5 titles
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)
        
        if result:
            match, score = result
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found match on Greenhouse: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match on Greenhouse: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found on Greenhouse (best: '{match}' at {score}%)"
        else:
            return "❌ No jobs matched that title in Greenhouse"

    except requests.exceptions.RequestException as e:
        debug_print(f"Request error: {e}")
        return f"❌ Error querying Greenhouse for '{slugged_company}': Network error"
    except json.JSONDecodeError as e:
        debug_print(f"JSON decode error: {e}")
        return f"❌ Error parsing Greenhouse response for '{slugged_company}'"
    except Exception as e:
        debug_print(f"Unexpected error: {e}")
        return f"❌ Unexpected error querying Greenhouse: {e}"

def validate_lever(slugged_company: str, job_title: str) -> str:
    """
    Search Lever API for job title
    """
    debug_print(f"Validating with Lever for company: {slugged_company}")
    
    url = f"https://jobs.lever.co/{slugged_company}?format=json"
    debug_print(f"Lever URL: {url}")
    
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        
        jobs = res.json()
        debug_print(f"Found {len(jobs)} jobs in Lever")
        
        if not jobs:
            return "❌ No jobs found in Lever feed"
        
        titles = [job.get('text', '') for job in jobs if job.get('text')]
        debug_print(f"Job titles found: {titles[:5]}...")  # Show first 5 titles
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)
        
        if result:
            match, score = result  # FIXED: was 'best_score'
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found match on Lever: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match on Lever: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found on Lever (best: '{match}' at {score}%)"
        else:
            return "❌ No jobs matched that title in Lever"

    except requests.exceptions.RequestException as e:
        debug_print(f"Request error: {e}")
        return f"❌ Error querying Lever for '{slugged_company}': Network error"
    except json.JSONDecodeError as e:
        debug_print(f"JSON decode error: {e}")
        return f"❌ Error parsing Lever response for '{slugged_company}'"
    except Exception as e:
        debug_print(f"Unexpected error: {e}")
        return f"❌ Unexpected error querying Lever: {e}"

def validate_workday(careers_url: str, job_title: str) -> str:
    """
    Validate job title against Workday API
    """
    debug_print(f"Validating with Workday for URL: {careers_url}")
    
    try:
        # Parse Workday URL to build API endpoint
        # Example: https://apple.wd5.myworkdayjobs.com/en-US/Apple_University_Talent
        # Becomes: https://apple.wd5.myworkdayjobs.com/wday/cxs/apple/Apple_University_Talent/jobs
        
        url_parts = careers_url.split("/")
        if len(url_parts) < 5:
            return "❌ Could not parse Workday URL structure"
        
        # Extract domain and path components
        domain = url_parts[2]  # e.g., "apple.wd5.myworkdayjobs.com"
        if len(url_parts) >= 6:
            path_slug = url_parts[5]  # Last part of the path
        else:
            path_slug = url_parts[4]
        
        # Extract company identifier from subdomain
        company_id = domain.split('.')[0]
        
        # Build API URL
        api_url = f"https://{domain}/wday/cxs/{company_id}/{path_slug}/jobs"
        debug_print(f"Workday API URL: {api_url}")

        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json"
        }
        
        res = requests.get(api_url, headers=headers, timeout=15)
        res.raise_for_status()
        
        data = res.json()
        jobs = data.get("jobPostings", [])
        debug_print(f"Found {len(jobs)} jobs in Workday")
        
        if not jobs:
            return "❌ No jobs found in Workday feed"

        titles = [job.get("title", "") for job in jobs if job.get("title")]
        debug_print(f"Job titles found: {titles[:5]}...")
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)

        if result:
            match, score = result
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found match in Workday: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in Workday: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found in Workday (best: '{match}' at {score}%)"
        else:
            return "❌ No match found in Workday"
            
    except requests.exceptions.RequestException as e:
        debug_print(f"Request error: {e}")
        return f"❌ Workday network error: {e}"
    except json.JSONDecodeError as e:
        debug_print(f"JSON decode error: {e}")
        return "❌ Could not parse Workday response"
    except Exception as e:
        debug_print(f"Unexpected error: {e}")
        return f"❌ Workday adapter error: {e}"

def validate_ashby(careers_url: str, job_title: str) -> str:
    """
    Validate job title against Ashby
    """
    debug_print(f"Validating with Ashby for URL: {careers_url}")
    
    try:
        res = requests.get(careers_url, timeout=15)
        res.raise_for_status()
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        # Look for Ashby cache data
        script = soup.find("script", string=re.compile("__ASHBY_CACHE__"))
        if not script:
            return "❌ Could not locate Ashby job data"
        
        # Extract JSON from script
        json_match = re.search(r'__ASHBY_CACHE__\s*=\s*(\{.*?\});', script.string, re.DOTALL)
        if not json_match:
            return "❌ Could not extract Ashby JSON"
        
        jobs_data = json.loads(json_match.group(1))
        jobs = jobs_data.get("jobs", [])
        debug_print(f"Found {len(jobs)} jobs in Ashby")
        
        if not jobs:
            return "❌ No jobs found in Ashby feed"
        
        titles = [job.get("title", "") for job in jobs if job.get("title")]
        debug_print(f"Job titles found: {titles[:5]}...")
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)
        
        if result:
            match, score = result
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found match in Ashby: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in Ashby: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found in Ashby (best: '{match}' at {score}%)"
        else:
            return "❌ No match found in Ashby"
            
    except Exception as e:
        debug_print(f"Error: {e}")
        return f"❌ Ashby adapter error: {e}"

def validate_smartrecruiters(careers_url: str, job_title: str) -> str:
    """
    Validate job title against SmartRecruiters
    """
    debug_print(f"Validating with SmartRecruiters for URL: {careers_url}")
    
    try:
        # Extract company identifier from URL
        match = re.search(r'smartrecruiters.com/([^/]+)', careers_url)
        if not match:
            return "❌ Could not parse SmartRecruiters URL"
        
        slugged_company = match.group(1)
        api_url = f"https://api.smartrecruiters.com/v1/companies/{slugged_company}/postings"
        debug_print(f"SmartRecruiters API URL: {api_url}")
        
        res = requests.get(api_url, timeout=15)
        res.raise_for_status()
        
        data = res.json()
        jobs = data.get("content", [])
        debug_print(f"Found {len(jobs)} jobs in SmartRecruiters")
        
        if not jobs:
            return "❌ No jobs found in SmartRecruiters feed"
        
        titles = [job.get("name", "") for job in jobs if job.get("name")]
        debug_print(f"Job titles found: {titles[:5]}...")
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)
        
        if result:
            match, score = result
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found match in SmartRecruiters: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match in SmartRecruiters: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found in SmartRecruiters (best: '{match}' at {score}%)"
        else:
            return "❌ No match found in SmartRecruiters"
            
    except Exception as e:
        debug_print(f"Error: {e}")
        return f"❌ SmartRecruiters adapter error: {e}"

def validate_icims(careers_url: str, job_title: str) -> str:
    """
    Validate job title against iCIMS
    """
    debug_print(f"Validating with iCIMS for URL: {careers_url}")
    
    try:
        # Try multiple iCIMS API patterns
        parsed_base = re.match(r"^(https://[\w.-]+/[^/]+)", careers_url)
        if not parsed_base:
            return "❌ Could not parse iCIMS base URL"

        base_url = parsed_base.group(1)
        possible_endpoints = [
            f"{base_url}/jobs/search.json",
            f"{base_url}/api/jobs",
            f"{careers_url.rstrip('/')}/search.json"
        ]
        
        for search_url in possible_endpoints:
            debug_print(f"Trying iCIMS endpoint: {search_url}")
            
            try:
                response = requests.get(search_url, timeout=10)
                response.raise_for_status()
                
                data = response.json()
                jobs = data.get("jobs", []) or data.get("data", [])
                
                if jobs:
                    debug_print(f"Found {len(jobs)} jobs in iCIMS")
                    titles = [job.get("title", "") for job in jobs if job.get("title")]
                    debug_print(f"Job titles found: {titles[:5]}...")
                    
                    sanitized_job_title = sanitize_title(job_title)
                    result = process.extractOne(sanitized_job_title, titles, scorer=fuzz.token_sort_ratio)
                    
                    if result:
                        match, score = result
                        debug_print(f"Best match: '{match}' with score: {score}")
                        
                        if score >= 85:
                            return f"✅ Found match in iCIMS: '{match}' ({score}%)"
                        elif score >= 70:
                            return f"⚠️ Partial match in iCIMS: '{match}' ({score}%)"
                        else:
                            return f"❌ No strong match found in iCIMS (best: '{match}' at {score}%)"
                    else:
                        return "❌ No match found in iCIMS"
                        
            except requests.exceptions.RequestException:
                continue  # Try next endpoint
        
        return "❌ Could not access iCIMS job data"

    except Exception as e:
        debug_print(f"Error: {e}")
        return f"❌ iCIMS adapter error: {e}"

def validate_simple_html(careers_url: str, job_title: str) -> str:
    """
    Validate job title by scraping HTML content
    """
    debug_print(f"Validating with HTML scraping for URL: {careers_url}")
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        
        res = requests.get(careers_url, headers=headers, timeout=15)
        res.raise_for_status()
        
        soup = BeautifulSoup(res.text, "html.parser")
        
        # Extract text from job-related elements
        job_elements = soup.find_all(["h1", "h2", "h3", "h4", "a", "div", "span"], 
                                   string=re.compile(r'\w+', re.I))
        
        texts = []
        for element in job_elements:
            text = element.get_text(strip=True)
            # Filter for reasonable job title length
            if 3 <= len(text.split()) <= 15:
                texts.append(text)
        
        debug_print(f"Found {len(texts)} potential job titles in HTML")
        debug_print(f"Sample texts: {texts[:10]}...")
        
        if not texts:
            return "❌ No job titles found in HTML content"
        
        sanitized_job_title = sanitize_title(job_title)
        result = process.extractOne(sanitized_job_title, texts, scorer=fuzz.token_sort_ratio)
        
        if result:
            match, score = result
            debug_print(f"Best match: '{match}' with score: {score}")
            
            if score >= 85:
                return f"✅ Found close match in HTML: '{match}' ({score}%)"
            elif score >= 70:
                return f"⚠️ Partial match found in HTML: '{match}' ({score}%)"
            else:
                return f"❌ No strong match found in HTML (best: '{match}' at {score}%)"
        else:
            return "❌ No match found in HTML content"
            
    except Exception as e:
        debug_print(f"Error: {e}")
        return f"❌ Error during HTML scrape: {e}"

def validate_with_selenium(careers_url: str, job_title: str) -> str:
    """
    Use Selenium as fallback for JavaScript-heavy sites
    """
    debug_print(f"Using Selenium fallback for URL: {careers_url}")
    
    driver = None
    try:
        options = ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        
        driver = webdriver.Chrome(
            service=ChromeService(ChromeDriverManager().install()), 
            options=options
        )
        
        driver.get(careers_url)
        
        # Wait for content to load
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Give JavaScript time to render
        time.sleep(3)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Extract job titles
        job_elements = soup.find_all